# Stage 1: Data Loading & Reading
Capstone Project: Spam Message Analysis Using Malicious URLs

Load the raw malicious URL dataset, inspect its shape, columns, data types and class distribution, and audit its data-quality problems before any cleaning.

In [1]:
from pathlib import Path

# Project root = parent of this notebook's stage folder (run the notebook from inside its own folder)
ROOT = Path.cwd().parent
RAW_CSV = ROOT / "dataset" / "dataset.csv"
INTER = ROOT / "dataset" / "intermediate"      # output of each stage becomes input of the next
RESULTS = ROOT / "results"
GRAPH_DIR = RESULTS / "graphs"
for p in (INTER, RESULTS, GRAPH_DIR):
    p.mkdir(parents=True, exist_ok=True)


In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set(style="whitegrid")
plt.rcParams['figure.figsize'] = (10, 6)

# Load raw dataset (encoding-safe: falls back gracefully if odd byte sequences are present)
FILE_PATH = RAW_CSV
try:
    df_raw = pd.read_csv(FILE_PATH)
except UnicodeDecodeError:
    df_raw = pd.read_csv(FILE_PATH, encoding="utf-8", encoding_errors="replace")

print("Shape:", df_raw.shape)
print("\nColumn names:", df_raw.columns.tolist())
print("\nData types:\n", df_raw.dtypes)
df_raw.head()

Shape: (651991, 2)

Column names: ['url', 'type']

Data types:
 url     str
type    str
dtype: object


,url,type
0,usa-people-search.com/Find-Jason-Bowser.aspx,benign
1,http://huangjintawujin.cn/examples/jsp/jsp2/js...,malware
2,kellysconfederateirishbrigade.com/battles/miss...,benign
3,http://www.easylivinggreenproducts.com.au/shop...,defacement
4,heavymetalmusicnews.com/,benign


In [3]:
# Quick look at the target classes present in the raw data
print("Class distribution (raw):")
print(df_raw["type"].value_counts())

Class distribution (raw):
type
benign        427942
defacement     96407
phishing       94087
malware        32505
BeNiGn           116
BENIGN           110
Benign           101
-                 47
NONE              46
123               43
phising           35
unknown           34
Defacement        32
DeFaCeMeNt        29
Phishing          24
DEFACEMENT        22
PHISHING          20
PhIsHiNg          15
MALWARE           11
MaLwArE           10
Malware           10
Name: count, dtype: int64


### Data Quality Audit — BEFORE Cleaning
Quantify every known issue in the raw file before touching it, so we have real before/after numbers for the review.

In [4]:
def data_quality_audit(data, label):
    print(f"===== DATA QUALITY AUDIT: {label} =====")
    print("Total rows:", len(data))
    print("Missing url:", data["url"].isna().sum())
    print("Missing type:", data["type"].isna().sum())
    print("Duplicate urls:", data["url"].duplicated().sum())

    non_null_url = data["url"].dropna().astype(str)
    print("URLs with leading/trailing whitespace:", (non_null_url != non_null_url.str.strip()).sum())

    non_null_type = data["type"].dropna().astype(str)
    print("Type values not already lowercase:", (non_null_type != non_null_type.str.lower()).sum())

    valid_lower = {"benign", "phishing", "defacement", "malware"}
    print("Invalid/garbage type labels:", (~non_null_type.str.strip().str.lower().isin(valid_lower)).sum())

    print("Empty-string or whitespace-only urls:", (non_null_url.str.strip().str.len() == 0).sum())
    print("Extremely long urls (>1000 chars):", (non_null_url.str.len() > 1000).sum())

    zero_width_pattern = r"[\u200b\ufeff]"
    print("URLs containing zero-width/BOM artifacts:", non_null_url.str.contains(zero_width_pattern, regex=True, na=False).sum())
    print()


In [5]:
data_quality_audit(df_raw, "RAW DATASET")

===== DATA QUALITY AUDIT: RAW DATASET =====
Total rows: 651991
Missing url: 329
Missing type: 345
Duplicate urls: 11354
URLs with leading/trailing whitespace: 553
Type values not already lowercase: 546
Invalid/garbage type labels: 205
Empty-string or whitespace-only urls: 46
Extremely long urls (>1000 chars): 39
URLs containing zero-width/BOM artifacts: 34

